In [3]:
#卷积层的计算，cross-correlation计算、padding、stride
import dnnlpy
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F #提供神经网络的运算函数，如relu、softmax、crossentropy
from torch import Tensor

dnnlpy.set_matplotlib_format('highdpi') #调用 dnnlpy 模块中的 set_matplotlib_format 方法，并将 highdpi 作为参数传进去。表示Jupyter绘图使用高清显示模式
print('PyTorch version:',torch.__version__)

PyTorch version: 2.14.0


In [5]:
#二维互相关计算函数
def corr2d(x:Tensor,kernel:Tensor)->Tensor:
    #input和kernel不是二维则无法计算
    if x.ndim!=2 or kernel.ndim!=2:
        raise AssertionError('input and kernel must both be 2d Tensor.')

    #定义卷积核的宽高K_h, K_w，output的宽高h，w
    K_h, K_w = kernel.size()
    h = x.size(0) - K_h + 1
    w = x.size(1) - K_w + 1

    #input宽高<= 0则错误
    if h <= 0 or w <= 0:
        raise RuntimeError('`kernel` must not be larger than the input.')

    #创建空output
    output=x.new_empty(h,w) #x.new_empty继承x的dtype和device

    #局部窗口和kernel的对应元素相乘并求和
    for i in range(h):
        for j in range(w):
            window=x[i:i+K_h,j:j+K_w]
            output[i,j]=torch.sum(kernel*window)

    return output

#测试函数corr2d
x=torch.arange(9,dtype=torch.float32).view(3,3)
kernel=torch.tensor([[0.0,1.0],
                     [2.0,3.0]])
y=corr2d(x,kernel)
print('Inout:',x,sep='\n') #sep='\n'：指定两个输出内容Input和x之间使用换行符分隔
print('Kernel:',kernel,sep='\n')
print('Output:',y,sep='\n')

Inout:
tensor([[0., 1., 2.],
        [3., 4., 5.],
        [6., 7., 8.]])
Kernel:
tensor([[0., 1.],
        [2., 3.]])
Output:
tensor([[19., 25.],
        [37., 43.]])


In [ ]:
#10*10图像经过卷积计算后输出8*8特征图
image = torch.zeros(10, 10)
image[:, 5:] = 1.0

kernel = torch.tensor(
    [
        [-1.0, 0.0, 1.0],
        [-1.0, 0.0, 1.0],
        [-1.0, 0.0, 1.0],
    ]
)
response = corr2d(image, kernel)

fig = plt.figure(1, figsize=(6, 3))
ax1 = fig.add_subplot(1, 2, 1)
ax1.imshow(image, cmap='gray', vmin=0, vmax=1)
ax1.set_xticks([])
ax1.set_yticks([])
ax1.set_title('Input')
ax2 = fig.add_subplot(1, 2, 2)
ax2.imshow(response, cmap='gray')
ax2.set_xticks([])
ax2.set_yticks([])
ax2.set_title('Vertical Edge Response')
plt.show()